In [ ]:
"""
Pipeline completo — Validación cruzada 10 folds
Carga .npy, aplica augmentation, calcula mel, entrena y evalúa.
"""

import os
import time
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import torchaudio.functional as F
import torchaudio.transforms as T
import urllib.request
import librosa
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, ConcatDataset

# ============================================================
# RUTAS
# ============================================================
RUTA_NPY = r'F:\UIS\2026-1\TRATAMIENTO DE SENALES DISCRETAS\PROYECTO\DATOS LIMPIOS Y A 4s\UrbanSound8K_4sec'

# ============================================================
# HIPERPARÁMETROS
# ============================================================
SR         = 22050
N_MELS     = 128
N_FFT      = 2048
HOP_LENGTH = 512
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR         = 0.001

# ============================================================
# AUGMENTATION
# ============================================================
_rir = None

def cargar_rir(ruta="rir_voces.wav"):
    global _rir
    if not os.path.exists(ruta):
        print("Descargando RIR...")
        url = "https://download.pytorch.org/torchaudio/tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav"
        urllib.request.urlretrieve(url, ruta)
    rir_raw, rir_sr = torchaudio.load(ruta)
    rir = rir_raw[:, int(rir_sr * 1.01): int(rir_sr * 1.3)]
    _rir = rir / torch.linalg.vector_norm(rir, ord=2)
    print("RIR listo.")

def _rir_aug(w):
    try:
        aug = F.fftconvolve(w, _rir)
        aug = aug[:, :w.shape[1]]
        m = aug.abs().max()
        return aug / m if m > 0 else aug
    except Exception:
        return w

def _add_noise_snr(w):
    snr_db = float(torch.empty(1).uniform_(3, 20).item())
    return F.add_noise(w, torch.randn_like(w), torch.tensor([snr_db]))

def _gaussian_noise(w):
    std = float(torch.empty(1).uniform_(0.002, 0.01).item())
    aug = w + torch.randn_like(w) * std
    m = aug.abs().max()
    return aug / m if m > 0 else aug

def _time_shift(w):
    max_m = int(0.5 * SR)
    shift = torch.randint(-max_m, max_m, (1,)).item()
    return torch.roll(w, shifts=shift, dims=1)

def _magnitude_warping(w):
    N      = w.shape[1]
    puntos = (1.0 + 0.6 * torch.randn(12)).unsqueeze(0).unsqueeze(0)
    curva  = torch.nn.functional.interpolate(
        puntos, size=N, mode="linear", align_corners=True
    ).squeeze()
    return w * curva.unsqueeze(0)

def _polarity_inversion(w):
    return w * -1

def _ganancia(w):
    factor = float(torch.empty(1).uniform_(0.3, 2.0).item())
    return (w * factor).clamp(-1, 1)

def augmentar(audio_np):
    w = torch.tensor(audio_np).float().unsqueeze(0)
    p = torch.rand(1).item()
    if p < 0.15:
        w = _rir_aug(w)
    elif p < 0.30:
        w = _add_noise_snr(w)
    elif p < 0.45:
        w = _gaussian_noise(w)
    elif p < 0.60:
        w = _time_shift(w)
    elif p < 0.75:
        w = _magnitude_warping(w)
    elif p < 0.90:
        w = _polarity_inversion(w)
    else:
        w = _ganancia(w)
    return w.squeeze(0).numpy()

# ============================================================
# DATASET
# ============================================================
class AudioDataset(Dataset):
    def __init__(self, X, Y, es_train=False):
        self.X        = X
        self.Y        = Y
        self.es_train = es_train

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        audio = self.X[idx]
        label = int(self.Y[idx])

        if self.es_train:
            audio = augmentar(audio)

        mel = librosa.feature.melspectrogram(
            y=audio, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
        )
        mel = librosa.power_to_db(mel, ref=np.max)
        mel = torch.tensor(mel).float().unsqueeze(0)  # (1, 128, T)

        return mel, label

# ============================================================
# MODELO
# ============================================================
class CNNNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.2)
        )
        self.pool    = nn.AdaptiveAvgPool2d((4, 4))
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024),
            nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.pool(x)
        x = self.flatten(x)
        return self.classifier(x)

# ============================================================
# ENTRENAMIENTO Y EVALUACIÓN
# ============================================================
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for mels, labels in loader:
        mels, labels = mels.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(mels)
        loss    = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluar(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for mels, labels in loader:
            mels, labels = mels.to(device), labels.to(device)
            outputs    = model(mels)
            loss       = loss_fn(outputs, labels)
            total_loss += loss.item()
            correct    += (outputs.argmax(dim=1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

# ============================================================
# CARGAR LOS 10 FOLDS
# ============================================================
cargar_rir()

print("\nCargando folds...")
folds_X, folds_Y = [], []
for i in range(1, 11):
    X = np.load(os.path.join(RUTA_NPY, f"fold{i}_X.npy"))
    Y = np.load(os.path.join(RUTA_NPY, f"fold{i}_Y.npy"))
    folds_X.append(X)
    folds_Y.append(Y)
    print(f"  fold{i}: {X.shape}")

# ============================================================
# VALIDACIÓN CRUZADA
# ============================================================
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDispositivo: {device}")

accuracies = []

for test_idx in range(10):
    print(f"\n{'='*55}")
    print(f"  Fold {test_idx+1} como prueba — {9} folds para entrenamiento")
    print(f"{'='*55}")

    # Dataset de prueba — sin augmentation
    test_ds = AudioDataset(folds_X[test_idx], folds_Y[test_idx], es_train=False)

    # 9 folds restantes — con augmentation
    train_ds = ConcatDataset([
        AudioDataset(folds_X[i], folds_Y[i], es_train=True)
        for i in range(10) if i != test_idx
    ])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # Modelo nuevo en cada fold
    model     = CNNNetwork().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn   = nn.CrossEntropyLoss()

    for epoch in range(NUM_EPOCHS):
        t = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        test_loss,  test_acc  = evaluar(model, test_loader, loss_fn, device)
        print(f"  Época {epoch+1:02d}/{NUM_EPOCHS} "
              f"| train_loss: {train_loss:.4f}  train_acc: {train_acc:.4f} "
              f"| test_loss: {test_loss:.4f}  test_acc: {test_acc:.4f} "
              f"| {time.time()-t:.1f}s")

    # Accuracy final del fold
    _, acc = evaluar(model, test_loader, loss_fn, device)
    accuracies.append(acc)
    print(f"\n  → Accuracy fold {test_idx+1}: {acc:.4f}")

# ============================================================
# RESULTADO FINAL
# ============================================================
print(f"\n{'='*55}")
print(f"  RESULTADO FINAL")
print(f"  Promedio: {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  Por fold: {[f'{a:.4f}' for a in accuracies]}")
print(f"{'='*55}")

# Boxplot
plt.figure(figsize=(6, 5))
plt.boxplot(accuracies, labels=["10-Fold CV"])
plt.ylabel("Accuracy")
plt.title("Validación cruzada 10 folds")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.show()

RIR listo.

Cargando folds...
  fold1: (768, 88200)
  fold2: (844, 88200)
  fold3: (846, 88200)
  fold4: (942, 88200)
  fold5: (920, 88200)
  fold6: (801, 88200)
  fold7: (815, 88200)
  fold8: (806, 88200)
  fold9: (816, 88200)
  fold10: (837, 88200)

Dispositivo: cuda

  Fold 1 como prueba — 9 folds para entrenamiento


c:\Users\molan\anaconda3\envs\pytorch-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
